<a href="https://colab.research.google.com/github/sohv/causality-in-llm/blob/main/LiNGAM_standard_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LiNGAM on ASIA

In [ ]:
!pip install pgmpy lingam pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.0/522.0 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.1 MB/s eta 0:00:00
   ━

In [ ]:
# load and simulate Asia dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/asia.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/8 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lingam/direct_lingam.py:224: RuntimeWarning: invalid value encountered in divide
  xj_std = (X[:, j] - np.mean(X[:, j])) / np.std(X[:, j])
/usr/local/lib/python3.11/dist-packages/lingam/direct_lingam.py:223: RuntimeWarning: invalid value encountered in divide
  xi_std = (X[:, i] - np.mean(X[:, i])) / np.std(X[:, i])
/usr/local/lib/python3.11/dist-packages/lingam/direct_lingam.py:151: RuntimeWarning: invalid value encountered in scalar divide
  return xi - (np.cov(xi, xj, bias=True)[0, 1] / np.var(xj)) * xj
/usr/local/lib/python3.11/dist-packages/lingam/direct_lingam.py:106: RuntimeWarning: invalid value encountered in cast
  X_[:, i] = self._residual(X_[:, i], X_[:, m])


In [ ]:
# ground Truth Adjacenct matrix
# =============================
ground_truth = np.zeros((8, 8), dtype=int)
ground_truth[node_indices['smoke']][node_indices['lung']] = 1
ground_truth[node_indices['smoke']][node_indices['bronc']] = 1
ground_truth[node_indices['asia']][node_indices['tub']] = 1
ground_truth[node_indices['lung']][node_indices['either']] = 1
ground_truth[node_indices['tub']][node_indices['either']] = 1
ground_truth[node_indices['either']][node_indices['xray']] = 1
ground_truth[node_indices['either']][node_indices['dysp']] = 1
ground_truth[node_indices['bronc']][node_indices['dysp']] = 1

In [ ]:
# evaluation
# ============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on Asia Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(8):
    for j in range(8):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(8):
    for j in range(8):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on Asia Dataset ===
Structured Hamming Distance: 13
True Positives: 1
False Positives: 6
False Negatives: 7
Precision: 0.143
Recall: 0.125
F1-Score: 0.133

=== Ground Truth Edges ===
  asia -> tub
  tub -> either
  smoke -> lung
  smoke -> bronc
  lung -> either
  bronc -> dysp
  either -> xray
  either -> dysp

=== Estimated Edges by LinGAM ===
  smoke -> lung
  bronc -> smoke
  either -> tub
  either -> lung
  xray -> either
  dysp -> bronc
  dysp -> either


In [ ]:
# graph
# =============
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG (Asia)", "ground_truth_asia.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_asia.png")

/tmp/ipython-input-5-2852178898.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


# LINGAM on CANCER

In [ ]:
# load and simulate cancer dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/cancer.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# ground truth for CANCER dataset
from pgmpy.readwrite import BIFReader
import numpy as np

# Load the Cancer BIF model
reader = BIFReader("/content/cancer.bif")
model = reader.get_model()

# Get node order and mapping
node_names = list(model.nodes())
node_indices = {name: i for i, name in enumerate(node_names)}
n_nodes = len(node_names)

# Initialize adjacency matrix
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill in edges
for u, v in model.edges():
    i, j = node_indices[u], node_indices[v]
    adj_matrix[i][j] = 1

# Display
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix (Cancer):")
print(adj_matrix)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i][j] == 1:
            print(f"{node_names[i]} -> {node_names[j]}")

Node order: ['Pollution', 'Smoker', 'Cancer', 'Xray', 'Dyspnoea']

Ground Truth Adjacency Matrix (Cancer):
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Edges:
Pollution -> Cancer
Smoker -> Cancer
Cancer -> Xray
Cancer -> Dyspnoea


In [ ]:
node_names = ['Pollution', 'Smoker', 'Cancer', 'Xray', 'Dyspnoea']
node_indices = {name: i for i, name in enumerate(node_names)}

# Initialize 5x5 adjacency matrix
ground_truth = np.zeros((5, 5), dtype=int)

# Define edges
ground_truth[node_indices['Pollution']][node_indices['Cancer']] = 1
ground_truth[node_indices['Smoker']][node_indices['Cancer']] = 1
ground_truth[node_indices['Cancer']][node_indices['Xray']] = 1
ground_truth[node_indices['Cancer']][node_indices['Dyspnoea']] = 1

In [ ]:
# evaluation
# ============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on Cancer Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(5):
    for j in range(5):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on Cancer Dataset ===
Structured Hamming Distance: 4
True Positives: 2
False Positives: 2
False Negatives: 2
Precision: 0.500
Recall: 0.500
F1-Score: 0.500

=== Ground Truth Edges ===
  Pollution -> Cancer
  Smoker -> Cancer
  Cancer -> Xray
  Cancer -> Dyspnoea

=== Estimated Edges by LinGAM ===
  Pollution -> Cancer
  Smoker -> Cancer
  Xray -> Cancer
  Dyspnoea -> Cancer


In [ ]:
# graph
# =============
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG (Asia)", "ground_truth_cancer.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_cancer.png")

/tmp/ipython-input-34-2859068134.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipython-input-34-2859068134.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


# LiNGAM on EARTHQUAKE

In [ ]:
# load and simulate Earthquake dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/earthquake.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# ground truth for Earthquake dataset
from pgmpy.readwrite import BIFReader
import numpy as np

# Load the Earthquake BIF model
reader = BIFReader("/content/earthquake.bif")
model = reader.get_model()

# Get node order and mapping
node_names = list(model.nodes())
node_indices = {name: i for i, name in enumerate(node_names)}
n_nodes = len(node_names)

# Initialize adjacency matrix
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill in edges
for u, v in model.edges():
    i, j = node_indices[u], node_indices[v]
    adj_matrix[i][j] = 1

# Display
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix (earthquake):")
print(adj_matrix)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i][j] == 1:
            print(f"{node_names[i]} -> {node_names[j]}")

Node order: ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']

Ground Truth Adjacency Matrix (earthquake):
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Edges:
Burglary -> Alarm
Earthquake -> Alarm
Alarm -> JohnCalls
Alarm -> MaryCalls


In [ ]:
node_names = ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']
node_indices = {name: i for i, name in enumerate(node_names)}

# Initialize 5x5 adjacency matrix
ground_truth = np.zeros((5, 5), dtype=int)

# Define edges
ground_truth[node_indices['Burglary']][node_indices['Alarm']] = 1
ground_truth[node_indices['Earthquake']][node_indices['Alarm']] = 1
ground_truth[node_indices['Alarm']][node_indices['JohnCalls']] = 1
ground_truth[node_indices['Alarm']][node_indices['MaryCalls']] = 1

# Print node order and adjacency matrix
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)

Node order: ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']

Ground Truth Adjacency Matrix:
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]


In [ ]:
# evaluation
# ============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on Earthquake Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(5):
    for j in range(5):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on Earthquake Dataset ===
Structured Hamming Distance: 7
True Positives: 1
False Positives: 4
False Negatives: 3
Precision: 0.200
Recall: 0.250
F1-Score: 0.222

=== Ground Truth Edges ===
  Burglary -> Alarm
  Earthquake -> Alarm
  Alarm -> JohnCalls
  Alarm -> MaryCalls

=== Estimated Edges by LinGAM ===
  Earthquake -> Burglary
  Earthquake -> Alarm
  Alarm -> Burglary
  JohnCalls -> Alarm
  MaryCalls -> Alarm


In [ ]:
# graph
# =============
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG (Asia)", "ground_truth_eq.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_eq.png")

/tmp/ipython-input-29-182271092.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipython-input-29-182271092.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


# LiNGAM on SACHS dataset

In [ ]:
# load and simulate Sachs dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/sachs.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
# ground truth for Sachs dataset
from pgmpy.readwrite import BIFReader
import numpy as np

# Load the Sachs BIF model
reader = BIFReader("/content/sachs.bif")
model = reader.get_model()

# Get node order and mapping
node_names = list(model.nodes())
node_indices = {name: i for i, name in enumerate(node_names)}
n_nodes = len(node_names)

# Initialize adjacency matrix
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill in edges
for u, v in model.edges():
    i, j = node_indices[u], node_indices[v]
    adj_matrix[i][j] = 1

# Display
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix (Sachs):")
print(adj_matrix)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i][j] == 1:
            print(f"{node_names[i]} -> {node_names[j]}")

Node order: ['Akt', 'Erk', 'Jnk', 'Mek', 'P38', 'PIP2', 'PIP3', 'PKA', 'PKC', 'Plcg', 'Raf']

Ground Truth Adjacency Matrix (Sachs):
[[0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [1 1 1 1 1 0 0 0 0 0 1]
 [0 0 1 1 1 0 0 1 0 0 1]
 [0 0 0 0 0 1 1 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0]]

Edges:
Erk -> Akt
Mek -> Erk
PIP3 -> PIP2
PKA -> Akt
PKA -> Erk
PKA -> Jnk
PKA -> Mek
PKA -> P38
PKA -> Raf
PKC -> Jnk
PKC -> Mek
PKC -> P38
PKC -> PKA
PKC -> Raf
Plcg -> PIP2
Plcg -> PIP3
Raf -> Mek


In [ ]:
# Define all unique nodes
node_names = ['Erk', 'Akt', 'Mek', 'PIP3', 'PIP2', 'PKA', 'Jnk', 'P38', 'Raf', 'PKC', 'Plcg']
node_indices = {name: i for i, name in enumerate(node_names)}

# Initialize adjacency matrix
n = len(node_names)
ground_truth = np.zeros((n, n), dtype=int)

# Define edges
edges = [
    ('Erk', 'Akt'),
    ('Mek', 'Erk'),
    ('PIP3', 'PIP2'),
    ('PKA', 'Akt'),
    ('PKA', 'Erk'),
    ('PKA', 'Jnk'),
    ('PKA', 'Mek'),
    ('PKA', 'P38'),
    ('PKA', 'Raf'),
    ('PKC', 'Jnk'),
    ('PKC', 'Mek'),
    ('PKC', 'P38'),
    ('PKC', 'PKA'),
    ('PKC', 'Raf'),
    ('Plcg', 'PIP2'),
    ('Plcg', 'PIP3'),
    ('Raf', 'Mek'),
]

# Fill the adjacency matrix
for parent, child in edges:
    i, j = node_indices[parent], node_indices[child]
    ground_truth[i][j] = 1

# Print node order and adjacency matrix
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)

Node order: ['Erk', 'Akt', 'Mek', 'PIP3', 'PIP2', 'PKA', 'Jnk', 'P38', 'Raf', 'PKC', 'Plcg']

Ground Truth Adjacency Matrix:
[[0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 1 1 1 0 0]
 [0 0 0 1 1 0 0 0 0 0 0]]


In [ ]:
# evaluation
# ==============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on Sachs Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(11):
    for j in range(11):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on Sachs Dataset ===
Structured Hamming Distance: 29
True Positives: 2
False Positives: 14
False Negatives: 15
Precision: 0.125
Recall: 0.118
F1-Score: 0.121

=== Ground Truth Edges ===
  Erk -> Akt
  Mek -> Erk
  PIP3 -> PIP2
  PKA -> Erk
  PKA -> Akt
  PKA -> Mek
  PKA -> Jnk
  PKA -> P38
  PKA -> Raf
  Raf -> Mek
  PKC -> Mek
  PKC -> PKA
  PKC -> Jnk
  PKC -> P38
  PKC -> Raf
  Plcg -> PIP3
  Plcg -> PIP2

=== Estimated Edges by LinGAM ===
  Erk -> Akt
  Erk -> Plcg
  PIP3 -> Erk
  PIP3 -> Akt
  PIP3 -> Raf
  PIP3 -> Plcg
  PIP2 -> P38
  PIP2 -> Raf
  Jnk -> PKA
  P38 -> Akt
  P38 -> Mek
  Raf -> Akt
  Raf -> P38
  PKC -> Akt
  PKC -> PKA
  Plcg -> P38


In [ ]:
# graph
# ====================
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG ", "ground_truth_sachs.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_sachs.png")

/tmp/ipython-input-24-2490414887.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipython-input-24-2490414887.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


# LiNGAM on Survey dataset

In [ ]:
# load and simulate Survey dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/survey.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# ground truth for survey dataset
from pgmpy.readwrite import BIFReader
import numpy as np

# Load the Survey BIF model
reader = BIFReader("/content/survey.bif")
model = reader.get_model()

# Get node order and mapping
node_names = list(model.nodes())
node_indices = {name: i for i, name in enumerate(node_names)}
n_nodes = len(node_names)

# Initialize adjacency matrix
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill in edges
for u, v in model.edges():
    i, j = node_indices[u], node_indices[v]
    adj_matrix[i][j] = 1

# Display
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix (Sachs):")
print(adj_matrix)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i][j] == 1:
            print(f"{node_names[i]} -> {node_names[j]}")

Node order: ['A', 'S', 'E', 'O', 'R', 'T']

Ground Truth Adjacency Matrix (Sachs):
[[0 0 1 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 1 0]
 [0 0 0 0 0 1]
 [0 0 0 0 0 1]
 [0 0 0 0 0 0]]

Edges:
A -> E
S -> E
E -> O
E -> R
O -> T
R -> T


In [ ]:
# Define node names and indices
node_names = ['A', 'S', 'E', 'O', 'R', 'T']
node_indices = {name: i for i, name in enumerate(node_names)}

# Initialize adjacency matrix
n = len(node_names)
ground_truth = np.zeros((n, n), dtype=int)

# Define edges
edges = [
    ('A', 'E'),
    ('S', 'E'),
    ('E', 'O'),
    ('E', 'R'),
    ('O', 'T'),
    ('R', 'T')
]

# Populate adjacency matrix
for parent, child in edges:
    i, j = node_indices[parent], node_indices[child]
    ground_truth[i][j] = 1

# Print results
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)

Node order: ['A', 'S', 'E', 'O', 'R', 'T']

Ground Truth Adjacency Matrix:
[[0 0 1 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 1 0]
 [0 0 0 0 0 1]
 [0 0 0 0 0 1]
 [0 0 0 0 0 0]]


In [ ]:
# evaluation
# ==============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on Survey Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(6):
    for j in range(6):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(6):
    for j in range(6):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on Survey Dataset ===
Structured Hamming Distance: 7
True Positives: 0
False Positives: 1
False Negatives: 6
Precision: 0.000
Recall: 0.000
F1-Score: 0.000

=== Ground Truth Edges ===
  A -> E
  S -> E
  E -> O
  E -> R
  O -> T
  R -> T

=== Estimated Edges by LinGAM ===
  T -> R


In [ ]:
# graph
# ====================
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG ", "ground_truth_survey.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_survey.png")

/tmp/ipython-input-40-1888659300.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


# LiNGAM on CHILD dataset

In [ ]:
# load and simulate CHILD dataset
# ================================================
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd
import numpy as np
from lingam import DirectLiNGAM

reader = BIFReader("/content/child.bif")
model = reader.get_model()

sampler = BayesianModelSampling(model)
simulated_data = sampler.forward_sample(size=1000, seed=42)

# Encode categorical variables numerically
df = simulated_data.copy()
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values
node_names = list(df.columns)
node_indices = {name: i for i, name in enumerate(node_names)}

# run LinGAM
# =============================
lingam_model = DirectLiNGAM()
lingam_model.fit(data)

# Adjacency matrix: 1 if there's a causal link
estimated_adj = (lingam_model.adjacency_matrix_ != 0).astype(int)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# ground truth for Child dataset
from pgmpy.readwrite import BIFReader
import numpy as np

# Load the Child BIF model
reader = BIFReader("/content/child.bif")
model = reader.get_model()

# Get node order and mapping
node_names = list(model.nodes())
node_indices = {name: i for i, name in enumerate(node_names)}
n_nodes = len(node_names)

# Initialize adjacency matrix
adj_matrix = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill in edges
for u, v in model.edges():
    i, j = node_indices[u], node_indices[v]
    adj_matrix[i][j] = 1

# Display
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix (Sachs):")
print(adj_matrix)

print("\nEdges:")
for i in range(n_nodes):
    for j in range(n_nodes):
        if adj_matrix[i][j] == 1:
            print(f"{node_names[i]} -> {node_names[j]}")

Node order: ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'CO2', 'ChestXray', 'Grunting', 'LVHreport', 'LowerBodyO2', 'RUQO2', 'CO2Report', 'XrayReport', 'Disease', 'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']

Ground Truth Adjacency Matrix (Sachs):
[[0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 0 

In [ ]:
# List of all unique nodes
node_names = [
    'BirthAsphyxia', 'Disease', 'HypDistrib', 'LowerBodyO2', 'HypoxiaInO2',
    'RUQO2', 'CO2', 'CO2Report', 'ChestXray', 'XrayReport', 'Grunting',
    'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing',
    'LungParench', 'LungFlow', 'Sick', 'LVHreport'
]

# Index mapping
node_indices = {name: i for i, name in enumerate(node_names)}
n = len(node_names)

# Initialize adjacency matrix
ground_truth = np.zeros((n, n), dtype=int)

# Define edges
edges = [
    ('BirthAsphyxia', 'Disease'),
    ('HypDistrib', 'LowerBodyO2'),
    ('HypoxiaInO2', 'LowerBodyO2'),
    ('HypoxiaInO2', 'RUQO2'),
    ('CO2', 'CO2Report'),
    ('ChestXray', 'XrayReport'),
    ('Grunting', 'GruntingReport'),
    ('Disease', 'Age'),
    ('Disease', 'LVH'),
    ('Disease', 'DuctFlow'),
    ('Disease', 'CardiacMixing'),
    ('Disease', 'LungParench'),
    ('Disease', 'LungFlow'),
    ('Disease', 'Sick'),
    ('LVH', 'LVHreport'),
    ('DuctFlow', 'HypDistrib'),
    ('CardiacMixing', 'HypDistrib'),
    ('CardiacMixing', 'HypoxiaInO2'),
    ('LungParench', 'HypoxiaInO2'),
    ('LungParench', 'CO2'),
    ('LungParench', 'ChestXray'),
    ('LungParench', 'Grunting'),
    ('LungFlow', 'ChestXray'),
    ('Sick', 'Grunting'),
    ('Sick', 'Age')
]

# Fill adjacency matrix
for parent, child in edges:
    i, j = node_indices[parent], node_indices[child]
    ground_truth[i][j] = 1

# Print node order
print("Node order:", node_names)

# Print adjacency matrix
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)

Node order: ['BirthAsphyxia', 'Disease', 'HypDistrib', 'LowerBodyO2', 'HypoxiaInO2', 'RUQO2', 'CO2', 'CO2Report', 'ChestXray', 'XrayReport', 'Grunting', 'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick', 'LVHreport']

Ground Truth Adjacency Matrix:
[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 1 0 

In [ ]:
# evaluation
# ==============
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    tp = np.sum(true_edges * est_edges)
    fp = np.sum((1 - true_edges) * est_edges)
    fn = np.sum(true_edges * (1 - est_edges))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

shd = calculate_shd(ground_truth, estimated_adj)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, estimated_adj)

print(f"\n=== LinGAM on CHILD Dataset ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

print(f"\n=== Ground Truth Edges ===")
for i in range(20):
    for j in range(20):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print(f"\n=== Estimated Edges by LinGAM ===")
for i in range(20):
    for j in range(20):
        if estimated_adj[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")


=== LinGAM on CHILD Dataset ===
Structured Hamming Distance: 59
True Positives: 2
False Positives: 36
False Negatives: 23
Precision: 0.053
Recall: 0.080
F1-Score: 0.063

=== Ground Truth Edges ===
  BirthAsphyxia -> Disease
  Disease -> Age
  Disease -> LVH
  Disease -> DuctFlow
  Disease -> CardiacMixing
  Disease -> LungParench
  Disease -> LungFlow
  Disease -> Sick
  HypDistrib -> LowerBodyO2
  HypoxiaInO2 -> LowerBodyO2
  HypoxiaInO2 -> RUQO2
  CO2 -> CO2Report
  ChestXray -> XrayReport
  Grunting -> GruntingReport
  LVH -> LVHreport
  DuctFlow -> HypDistrib
  CardiacMixing -> HypDistrib
  CardiacMixing -> HypoxiaInO2
  LungParench -> HypoxiaInO2
  LungParench -> CO2
  LungParench -> ChestXray
  LungParench -> Grunting
  LungFlow -> ChestXray
  Sick -> Grunting
  Sick -> Age

=== Estimated Edges by LinGAM ===
  BirthAsphyxia -> Disease
  HypDistrib -> Disease
  HypDistrib -> CO2Report
  HypDistrib -> CardiacMixing
  HypDistrib -> Sick
  LowerBodyO2 -> XrayReport
  LowerBodyO2 -> 

In [ ]:
# graph
# ====================
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj_matrix, title, filename):
    G = nx.DiGraph()
    for i in range(len(node_names)):
        for j in range(len(node_names)):
            if adj_matrix[i][j] == 1:
                G.add_edge(node_names[i], node_names[j])

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000,
            font_size=10, arrows=True, arrowstyle='->', arrowsize=15)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

draw_graph(ground_truth, "Ground Truth DAG ", "ground_truth_child.png")
draw_graph(estimated_adj, "Estimated DAG (LinGAM)", "lingam_estimated_child.png")

/tmp/ipython-input-46-3633552868.py:18: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
